In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

import pandas as pd

In [16]:
data = pd.read_csv('./dataset/USR.csv')
data['date'] = data['Year'].astype(str) + '-' + data['Month'].astype(str)
data.set_index('date', inplace=True)

In [3]:
def create_lags(df, k, value_col):
    """
    Transforma a série temporal em um DataFrame supervisionado.

    k: quantidade de lags
    value_col: nome da coluna com os valores numéricos
    """

    data = df[value_col].copy()

    result = pd.DataFrame()

    for i in range(k, 0, -1):
        result[f'x-{i}'] = data.shift(i)

    result['x'] = data

    result = result.dropna().reset_index(drop=True)

    return result

In [ ]:
def fitness(params, df, value_col, test_size=0.2):
    """
    Fitness para otimização dos hiperparâmetros de uma MLP
    utilizando PSO.

    params[0]: quantidade de lags (k)
    params[1]: neurônios na camada escondida (n)

    Retorna o MSE de validação.
    """

    k = int(round(params[0]))
    n = int(round(params[1]))

    # Evita configurações inválidas
    if k < 1 or n < 1:
        return 1e10

    # Transformação supervisionada
    data = create_lags(df, k, value_col)

    X = data.drop(columns=['x']).values
    y = data['x'].values

    # Divisão cronológica
    split = int(len(X) * (1 - test_size))

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    if len(X_train) <= k or len(X_test) == 0:
        return 1e10

    # Normalização
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train = scaler_X.fit_transform(X_train)
    X_test = scaler_X.transform(X_test)

    y_train = scaler_y.fit_transform(
        y_train.reshape(-1, 1)
    ).ravel()

    # Arquitetura da MLP
    model = MLPRegressor(
        hidden_layer_sizes=(n,),
        activation='relu',
        solver='adam',
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=30
    )

    # Treinamento
    model.fit(X_train, y_train)

    # Previsão
    predictions = model.predict(X_test)

    # Retorna à escala original
    predictions = scaler_y.inverse_transform(
        predictions.reshape(-1, 1)
    ).ravel()

    # Erro de validação
    mse = mean_squared_error(y_test, predictions)

    return mse